# 02 · LCEL, Runnables and output parsers

After this notebook you can read and write LangChain Expression Language (LCEL): compose `prompt | model | parser`,
add parallel steps and passthrough fields, route between chains, wrap steps in retries and fallbacks, batch and stream
a whole chain, and inspect it as a graph with tags and callbacks.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Models, messages and prompts](01_models_messages_and_prompts.ipynb)

## Why this matters in interviews

- Every LangChain pipeline — RAG, extraction, routing — is written in LCEL. "Read this chain and tell me what flows
  between the steps" is a common live exercise.
- Production follow-ups map one-to-one onto LCEL features: retries with backoff, fallback models, batching and
  concurrency, streaming, tracing. You should know the method name *and* the trap for each.
- Routing (classify, then send to the right chain) is the cheapest pattern with the biggest quality win.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pr23` | What is prompt chaining and when is it better than one big prompt? |
| `pr07` | How do you get reliable JSON out of a model? |
| `ag31` | What is a router agent, and why is it the highest-leverage pattern? |
| `in28` | What is a fallback model strategy? |
| `pd13` | What is a fallback model strategy and how do you test it? |
| `po16` | How do you design a retry policy that does not make things worse? |
| `in29` | What is the difference between concurrency, batch size and throughput? |
| `pd39` | What does "streaming" mean in a production LLM system, and what are the gotchas? |
| `tl05` | How would you add observability to an LLM application? |

## Setup

`langchain_chat_model()` is `ChatOpenAI(model="gpt-4.1-mini")` wired to whichever provider is active (OpenAI, Ollama or
the offline stand-in). The stand-in does arithmetic, classification, summaries and JSON-schema filling
deterministically, so the chains below give the same output every run.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model, OFFLINE

import time
import matplotlib.pyplot as plt
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel, RunnableBranch

llm = langchain_chat_model(temperature=0)

## 1. The Runnable interface and the `|` operator

**In plain English:** every building block — a prompt, a model, a parser, a retriever, your own function — is a
*Runnable*: a box with the same buttons. `a | b` means "feed a's output into b", and the result is again a Runnable
with the same buttons, so a whole pipeline behaves like one component.

| Method | Does | Async twin |
|---|---|---|
| `invoke(x)` | one input → one output | `ainvoke` |
| `batch([x1, x2, ...])` | many inputs, run concurrently, results in input order | `abatch` |
| `stream(x)` | yield output chunks as they are produced | `astream`, `astream_events` |
| `with_retry()`, `with_fallbacks()`, `with_config()`, `bind()` | wrap the runnable, return a new one | — |
| `get_graph()`, `input_schema`, `output_schema` | introspection | — |

Coercion rules: a **dict** inside a chain becomes a `RunnableParallel`, a **function** becomes a `RunnableLambda`.

In [ ]:
prompt = ChatPromptTemplate.from_messages([("system", "You are a calculator."), ("human", "What is {expr}?")])
chain = prompt | llm | StrOutputParser()

print("type        :", type(chain).__name__)
print("steps       :", [type(s).__name__ for s in chain.steps])
print("input schema:", chain.input_schema.model_json_schema()["properties"])
print("answer      :", chain.invoke({"expr": "12*12"}))

What flows between the steps? Run the pieces one at a time: a dict goes in, the prompt makes a `ChatPromptValue`,
the model makes an `AIMessage`, the parser makes a `str`. Knowing the type at every hop is how you debug a chain.

In [ ]:
x0 = {"expr": "12*12"}
x1 = prompt.invoke(x0)
x2 = llm.invoke(x1)
x3 = StrOutputParser().invoke(x2)
for name, val in [("input", x0), ("after prompt", x1), ("after model", x2), ("after parser", x3)]:
    kind = type(val).__name__ + (" (a str subclass)" if isinstance(val, str) and type(val) is not str else "")
    print(f"{name:13s} {kind:31s} {str(val)[:60]!r}")
assert isinstance(x3, str) and x3 == chain.invoke(x0)

chain.get_graph().print_ascii()

## 2. Output parsers

| Parser | Turns the model output into | Validates a schema? |
|---|---|---|
| `StrOutputParser` | the message text | no |
| `JsonOutputParser` | a Python dict/list; strips ```json fences; streams **partial** JSON | no — any valid JSON passes |
| `JsonOutputParser(pydantic_object=M)` / `PydanticOutputParser(M)` | a dict / a validated `M` instance; adds format instructions for the prompt | Pydantic version: yes |
| `llm.with_structured_output(M)` (notebook 03) | a validated `M`, using the API's native structured output | yes, enforced by the provider |

Parsers are Runnables too, so you can test them without a model:

In [ ]:
from langchain_core.exceptions import OutputParserException
from langchain_core.outputs import Generation

jp = JsonOutputParser()
print(jp.invoke('```json\n{"city": "Paris", "temp_c": 18}\n```'))          # markdown fences are stripped
half_streamed = [Generation(text='{"city": "Paris", "temp_c": 1')]           # a stream cut mid-object
print(jp.parse_result(half_streamed, partial=True))                         # ...still parses (partial=True)
try:
    jp.invoke("Sure! The city is Paris.")
except OutputParserException as e:
    print("not JSON ->", type(e).__name__, "|", str(e).splitlines()[0][:60])

Asking a model for JSON in the prompt alone is fragile ("Sure! Here is the JSON: ..."). The reliable route is to make
the API constrain the output: here we `bind` an OpenAI `response_format` JSON schema to the model, then parse. Because
`JsonOutputParser` understands partial JSON, **streaming** the chain yields a growing dict — useful for filling a UI
field by field.

In [ ]:
person_schema = {"name": "person", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["name", "age", "city"],
    "properties": {"name": {"type": "string"}, "age": {"type": "integer"}, "city": {"type": "string"}}}}
json_llm = llm.bind(response_format={"type": "json_schema", "json_schema": person_schema})

extract = ChatPromptTemplate.from_messages([("system", "Extract the person."), ("human", "{text}")]) | json_llm | JsonOutputParser()
text = "Alice is 31 and lives in Paris."
print("invoke:", extract.invoke({"text": text}))
for partial in extract.stream({"text": text}):
    print("  stream ->", partial)
assert extract.invoke({"text": text}) == {"name": "Alice", "age": 31, "city": "Paris"}

## 3. `RunnableLambda`: your own Python as a step

Any function of **one argument** can be a step (wrap it, or just pipe it — it is coerced). Multiple inputs arrive as
one dict. The `@chain` decorator turns a function into a Runnable directly.

In [ ]:
from langchain_core.runnables import chain as as_runnable

def clean(text: str) -> str:
    return " ".join(text.split()).strip()

@as_runnable
def word_count(text: str) -> dict:
    return {"text": text, "words": len(text.split())}

pipeline = RunnableLambda(clean) | word_count
print(pipeline.invoke("   LCEL   turns\n functions   into   steps  "))
print("batch:", pipeline.batch(["a b", "  c  d  e "]))

## 4. `RunnablePassthrough` and `.assign`: keep the input, add fields

**In plain English:** most real chains need the original input *and* the model's output at the end (the question and
the answer, the document and its summary). `RunnablePassthrough.assign(key=runnable)` runs the runnable on the input
dict and **adds** its result under `key`, keeping everything else.

In [ ]:
answer_chain = prompt | llm | StrOutputParser()

enrich = (
    RunnablePassthrough.assign(n_chars=lambda d: len(d["expr"]))
    .assign(answer=answer_chain)               # sees {"expr": ..., "n_chars": ...}
    .assign(ok=lambda d: d["answer"].endswith("."))
)
out = enrich.invoke({"expr": "17*23"})
print(out)
assert set(out) == {"expr", "n_chars", "answer", "ok"} and "391" in out["answer"]

## 5. `RunnableParallel`: fan out on the same input

Two independent LLM calls on the same input should run **at the same time** — the latency is the slowest branch, not
the sum. A dict literal in a chain is the same thing (`{"summary": a, "sentiment": b}`). To make the timing visible
offline we give the stand-in 0.3 s of network latency.

In [ ]:
summarize = ChatPromptTemplate.from_messages([("human", "Summarize this review.\n\n{review}")]) | llm | StrOutputParser()
sentiment = (ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment as one of: positive, negative, neutral."), ("human", "{review}")])
    | llm | StrOutputParser())
both = RunnableParallel(summary=summarize, sentiment=sentiment)
review = {"review": "I love this laptop. The battery lasts all day. The screen is great."}

if OFFLINE:
    OFFLINE.latency = 0.3
t0 = time.perf_counter(); result = both.invoke(review); t_parallel = time.perf_counter() - t0
t0 = time.perf_counter(); summarize.invoke(review); sentiment.invoke(review); t_serial = time.perf_counter() - t0
if OFFLINE:
    OFFLINE.latency = 0.0

print(result)
print(f"parallel {t_parallel:.2f}s vs one after the other {t_serial:.2f}s")
assert result["sentiment"] == "positive"
if OFFLINE:
    assert t_parallel < 0.8 * t_serial

## 6. Routing: classify, then send to the right chain

**In plain English:** a small, cheap first call decides what kind of request this is; each kind gets its own prompt
(or model, or tool). This is the router pattern (`ag31`): specialised prompts are simpler and better than one prompt
that handles everything.

Two ways to route in LCEL:

- `RunnableBranch((condition, runnable), ..., default)` — first condition that is true wins.
- **A function that returns a runnable** (recommended by the LangChain docs): LCEL runs whatever runnable the
  function returns. Plain Python, easy to test.

In [ ]:
classify = (ChatPromptTemplate.from_messages([
    ("system", "Classify the user's message into one of: general, refunds, login. Reply with the label only."),
    ("human", "{question}")]) | llm | StrOutputParser())

refund_chain = RunnableLambda(lambda x: f"[refund team] {x['question']}")
login_chain = RunnableLambda(lambda x: f"[login help] {x['question']}")
general_chain = RunnableLambda(lambda x: f"[general FAQ] {x['question']}")

branch = RunnableBranch(
    (lambda x: "refund" in x["topic"].lower(), refund_chain),
    (lambda x: "login" in x["topic"].lower(), login_chain),
    general_chain,                                           # default
)
router = RunnablePassthrough.assign(topic=classify) | branch

def route(x: dict):
    topic = x["topic"].strip().lower()
    return {"refunds": refund_chain, "login": login_chain}.get(topic, general_chain)

router_fn = RunnablePassthrough.assign(topic=classify) | RunnableLambda(route)

questions = ["I want a refund for my order", "I forgot my password", "What are your opening hours?"]
for q in questions:
    a, b = router.invoke({"question": q}), router_fn.invoke({"question": q})
    print(f"{q:32s} -> {a}")
    assert a == b
assert router.invoke({"question": questions[1]}).startswith("[login help]")

The graph shows the shape: the classifier runs in parallel with a passthrough of the input, then the branch.

In [ ]:
router.get_graph().print_ascii()

## 7. Fallbacks and retries

| Tool | What it does | Use it for |
|---|---|---|
| `r.with_retry(retry_if_exception_type=(...), stop_after_attempt=3, wait_exponential_jitter=True)` | re-runs `r` on the listed errors, with exponential backoff + jitter | transient errors: 429, timeouts, 5xx |
| `r.with_fallbacks([backup, ...], exceptions_to_handle=(...))` | if `r` raises, try each backup in order | provider outage, a model that is down, a cheaper/bigger model as plan B |
| `ChatOpenAI(max_retries=2)` | the OpenAI SDK's own retries (honours `Retry-After`) | on by default: **2** retries |

The stand-in can fail on purpose: `OFFLINE.fail_next(n, status)` makes the next `n` requests return that HTTP status.
First a fallback: the primary asks for a model that does not exist (a real API answers 404; offline we inject the
404), and the backup model answers.

In [ ]:
primary = langchain_chat_model(model="gpt-does-not-exist", max_retries=0)
backup = langchain_chat_model(model="gpt-4.1-nano", max_retries=0)
resilient = primary.with_fallbacks([backup])

if OFFLINE:
    OFFLINE.fail_next(1, 404)
reply = resilient.invoke("What is 6*7?")
print("answer from:", reply.response_metadata["model_name"], "->", reply.content)
assert reply.response_metadata["model_name"] == "gpt-4.1-nano"

Now a retry: two 429s ("rate limited"), then success. Retry only on errors that can succeed next time — a 400 (bad
request) will fail identically on every attempt, so retrying it just burns money and time.

In [ ]:
no_sdk_retry = langchain_chat_model(max_retries=0)
retrying = no_sdk_retry.with_retry(
    retry_if_exception_type=(openai.RateLimitError, openai.APITimeoutError, openai.InternalServerError),
    stop_after_attempt=3, wait_exponential_jitter=True,
    exponential_jitter_params={"initial": 0.1, "max": 1.0, "jitter": 0.1})

if OFFLINE:
    OFFLINE.fail_next(2, 429)
    sent_before = OFFLINE.request_count
print(retrying.invoke("What is 2+2?").content)
if OFFLINE:
    print("requests sent:", OFFLINE.request_count - sent_before, "(two 429s, then success)")
    assert OFFLINE.request_count - sent_before == 3

**The retry-storm trap (`po16`).** Retry layers multiply. The SDK already retries twice; wrap that in `with_retry(3)`
and one user request can become 3 × 3 = 9 API calls — exactly when the provider is overloaded. Pick **one** layer
(usually the SDK's, or yours with `max_retries=0` on the model), cap total attempts, and add jitter so clients do not
retry in lock-step. (This demo only makes sense with injected failures, so it runs offline only.)

In [ ]:
stacked = langchain_chat_model(max_retries=2).with_retry(
    stop_after_attempt=3, exponential_jitter_params={"initial": 0.05, "max": 0.1, "jitter": 0.05})
if OFFLINE:
    OFFLINE.fail_next(9, 429)
    sent_before = OFFLINE.request_count
    try:
        stacked.invoke("What is 2+2?")
    except openai.RateLimitError:
        print("gave up after", OFFLINE.request_count - sent_before, "API requests for ONE invoke")
    assert OFFLINE.request_count - sent_before == 9

## 8. `batch` vs a loop vs `abatch`

LLM calls spend almost all their time waiting on the network, so running them concurrently is nearly free speed.
`batch` uses a thread pool; `abatch` uses asyncio; `config={"max_concurrency": n}` caps how many run at once (your
real limit is the provider's rate limit, not your CPU). Offline we set 0.3 s of latency per request.

To *see* the concurrency, a small callback handler records when each model call starts and ends (callbacks are
covered in section 10).

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class SpanRecorder(BaseCallbackHandler):
    """Records (start, end) of every model call, in seconds since the recorder was created."""
    def __init__(self):
        self.t0, self.open, self.spans = time.perf_counter(), {}, []
    def on_chat_model_start(self, serialized, messages, *, run_id, **kw):
        self.open[run_id] = time.perf_counter() - self.t0
    def on_llm_end(self, response, *, run_id, **kw):
        self.spans.append((self.open.pop(run_id), time.perf_counter() - self.t0))

qs = [f"What is {i}+{i}?" for i in range(8)]
calc = llm | StrOutputParser()
timings, spans, outputs = {}, {}, {}

def measure(label, fn):
    rec = SpanRecorder()
    t0 = time.perf_counter(); outputs[label] = fn({"callbacks": [rec]}); timings[label] = time.perf_counter() - t0
    spans[label] = sorted(rec.spans)

if OFFLINE:
    OFFLINE.latency = 0.3
measure("for loop", lambda cfg: [calc.invoke(q, config=cfg) for q in qs])
measure("batch, max_concurrency=2", lambda cfg: calc.batch(qs, config={**cfg, "max_concurrency": 2}))
measure("batch", lambda cfg: calc.batch(qs, config=cfg))
t0 = time.perf_counter(); outputs["abatch"] = await calc.abatch(qs); timings["abatch"] = time.perf_counter() - t0
if OFFLINE:
    OFFLINE.latency = 0.0

assert all(out == outputs["for loop"] for out in outputs.values())     # same answers, same order
for k, v in timings.items():
    print(f"{k:25s} {v:.2f}s  -> {len(qs) / v:5.1f} requests/s")
if OFFLINE:
    assert timings["batch"] < timings["batch, max_concurrency=2"] < timings["for loop"]
    assert timings["batch"] < timings["for loop"] / 3

In [ ]:
plt.figure(figsize=(8, 3.2))
plt.bar(list(timings), list(timings.values()), color=["tab:gray", "tab:orange", "tab:blue", "tab:green"])
plt.title(f"Wall time for {len(qs)} requests"); plt.ylabel("seconds")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for ax, (label, sp) in zip(axes, spans.items()):
    for i, (start, end) in enumerate(sp):
        ax.barh(i, end - start, left=start, height=0.6, color="tab:blue")
    ax.set_title(label); ax.set_xlabel("seconds since start")
axes[0].set_ylabel("request #"); axes[0].invert_yaxis()
plt.suptitle("When each request was in flight: concurrency = overlapping bars")
plt.tight_layout(); plt.show()

## 9. Streaming through a chain

`chain.stream()` streams if **every step can pass chunks through**. Prompt templates, chat models and
`StrOutputParser` / `JsonOutputParser` can. A plain function cannot — `RunnableLambda` waits for its whole input, so
one lambda at the end silently turns your stream into a single chunk. The fix is a **generator** that takes an
iterator of chunks and yields transformed chunks.

In [ ]:
story = ChatPromptTemplate.from_messages([("human", "Summarize this text.\n\n{text}")]) | llm | StrOutputParser()
payload = {"text": "LCEL composes runnables with the pipe operator. Every runnable has invoke, batch and stream. "
                   "Streaming shows tokens early."}

def shout_stream(chunks):              # generator: iterator of chunks in, chunks out
    for c in chunks:
        yield c.upper()

n_plain = len(list(story.stream(payload)))
n_lambda = len(list((story | RunnableLambda(lambda s: s.upper())).stream(payload)))
n_gen = len(list((story | shout_stream).stream(payload)))
print(f"chunks: plain chain {n_plain} | + lambda {n_lambda} | + generator {n_gen}")
print("first chunks via generator:", list((story | shout_stream).stream(payload))[:5])
assert n_lambda == 1 and n_gen == n_plain > 1

## 10. Config: names, tags, metadata and callbacks

**In plain English:** tags and names are labels on each step, so a trace of a production request tells you which
chain, which version and which user it belonged to. Every invoke accepts a `config` dict:

| Key | Effect |
|---|---|
| `run_name` | the name this run shows under in traces |
| `tags` | labels, **inherited by every child step** (e.g. `["prod", "v2"]`) |
| `metadata` | key/values attached to the run (user id, tenant, experiment) |
| `callbacks` | handlers notified on start/end/error of every step — this is how LangSmith and other tracers hook in |
| `max_concurrency` | cap for `batch` |

`.with_config(...)` bakes config into a runnable. Here is a tiny callback handler that prints the trace tree.

In [ ]:
class PrintTracer(BaseCallbackHandler):
    def on_chain_start(self, serialized, inputs, *, parent_run_id=None, tags=None, metadata=None, **kw):
        print(f"{'  ' if parent_run_id else ''}start {kw.get('name')!s:20s} tags={tags} user={metadata.get('user_id')}")
    def on_chat_model_start(self, serialized, messages, *, tags=None, **kw):
        print(f"  start {'ChatOpenAI':20s} tags={tags}")
    def on_llm_end(self, response, **kw):
        print(f"  llm tokens: {response.generations[0][0].message.usage_metadata['total_tokens']}")

named = answer_chain.with_config(run_name="calculator_chain", tags=["v2"])
named.invoke({"expr": "3+4"}, config={"callbacks": [PrintTracer()], "tags": ["prod"], "metadata": {"user_id": "u-7"}})

## Try it yourself

**1. No-model pipeline.** Using only `RunnableParallel`, `RunnablePassthrough` and lambdas, build a chain that turns
`"  Hello   LCEL world "` into `{"original": ..., "clean": "Hello LCEL world", "words": 3}`.

In [ ]:
# Solution — try it yourself first, then run this
tidy = RunnableParallel(original=RunnablePassthrough(), clean=RunnableLambda(clean)) | RunnablePassthrough.assign(
    words=lambda d: len(d["clean"].split()))
res = tidy.invoke("  Hello   LCEL world ")
print(res)
assert res == {"original": "  Hello   LCEL world ", "clean": "Hello LCEL world", "words": 3}

**2. A math router.** Send questions that contain an arithmetic expression to a local Python calculator (no LLM call)
and everything else to the `general_chain`. Check both routes.

In [ ]:
# Solution — try it yourself first, then run this
import re

def local_calc(x: dict) -> str:
    expr = re.search(r"[\d\s\+\-\*/\(\)\.]*\d[\d\s\+\-\*/\(\)\.]*", x["question"]).group().strip()
    return f"[calculator] {expr} = {eval(expr, {'__builtins__': {}})}"    # demo only: never eval untrusted input

def math_or_general(x: dict):
    return RunnableLambda(local_calc) if re.search(r"\d\s*[\+\-\*/]\s*\d", x["question"]) else general_chain

math_router = RunnableLambda(math_or_general)
print(math_router.invoke({"question": "What is 12 * 7?"}))
print(math_router.invoke({"question": "Do you ship to Canada?"}))
assert math_router.invoke({"question": "What is 12 * 7?"}).endswith("= 84")

**3. Tell the fallback what went wrong.** `with_fallbacks(..., exception_key="error")` passes the exception to the
backup inside the input dict. Build a primary that always raises and a backup that reports the error type.

In [ ]:
# Solution — try it yourself first, then run this
def always_down(d: dict) -> str:
    raise TimeoutError("primary timed out")

safe = RunnableLambda(always_down).with_fallbacks(
    [RunnableLambda(lambda d: f"served by backup after {type(d['error']).__name__}")], exception_key="error")
msg = safe.invoke({"question": "hi"})
print(msg)
assert msg == "served by backup after TimeoutError"

## Say it in an interview

- **LCEL in one sentence:** every component is a Runnable with `invoke / batch / stream` (+ async); `|` composes them
  into a Runnable, so retries, fallbacks, batching, streaming and tracing work on the whole pipeline for free.
- **Know the hops:** dict → prompt → `ChatPromptValue` → model → `AIMessage` → parser → `str`/dict. Debug by
  invoking steps one at a time.
- **Shapes:** `RunnablePassthrough.assign` keeps the input and adds fields (question + answer + sources);
  `RunnableParallel` (or a dict) fans out — latency is the slowest branch, not the sum; route with a function that
  returns a runnable (or `RunnableBranch`).
- **JSON:** prompt-only JSON is fragile; constrain it at the API (JSON schema / `with_structured_output`) and parse.
  `JsonOutputParser` accepts any valid JSON and streams partial objects; it does not validate your schema.
- **Resilience:** retry only transient errors (429, timeout, 5xx) with exponential backoff + jitter and a cap; never
  retry 400s; do not stack retry layers (2 SDK retries × 3 attempts = 9 calls); fallbacks for outages, and test them by
  injecting failures.
- **Throughput:** `batch`/`abatch` with `max_concurrency` — LLM calls are I/O-bound, the real ceiling is the provider's
  rate limit.
- **Streaming trap:** a plain function step blocks the stream; use a generator step.
- **Observability:** `run_name`, `tags`, `metadata` and callbacks propagate to child runs — that is what LangSmith and
  OpenTelemetry exporters read.

## Next

- [03 · Structured output and tools](03_structured_output_and_tools.ipynb)
- [05 · The RAG chain end to end](05_rag_chain_end_to_end.ipynb) — LCEL's most famous chain.
- Without a framework: [async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb),
  [retries, backoff, rate limits and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb),
  [observability, logging and tracing](../16_production/08_observability_logging_and_tracing.ipynb)
- Theory: [LangChain course](../../langchain/index.html) (chapter 4, LCEL) · [interview bank](../../ai_interview_prep/index.html)